# ContractNLI - Milestone 1 Inference Notebook

Base model: `unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit`  
LoRA adapter: `./kaggle/working/qwen3-4B-nli-lora-adapter`

## 0. Install Dependencies

In [86]:
# !pip install -q "pyarrow==15.0.2" --force-reinstall
# !pip install -q unsloth 
# kagglehub unsloth peft transformers accelerate bitsandbytes
# # *** RESTART THE KERNEL AFTER RUNNING THIS CELL ***
!pip install -q json-repair unsloth

In [87]:
# Mount Google Drive and extract the adapter zip
# Skip this cell if the adapter is already extracted
# from google.colab import drive
# drive.mount("/content/drive")

# import zipfile, os

# ZIP_PATH = "/content/drive/MyDrive/qwen-nli-lora-adapter (1).zip"
# EXTRACT_TO = "/content/"

# if not os.path.isdir("/content/qwen3-4B-nli-lora-adapter"):
#     print(f"Extracting {ZIP_PATH}...")
#     with zipfile.ZipFile(ZIP_PATH) as zf:
#         zf.extractall(EXTRACT_TO)
#     print("Done.")
# else:
#     print("Adapter already extracted, skipping.")

# print("Adapter files:", os.listdir("/content/qwen3-4B-nli-lora-adapter"))

## 1. Dataset Download & Preparation

Taken from the training notebook (cells 0–11). Downloads ContractNLI from
Kaggle, explores the file structure, inspects document/annotation format,
analyses label & evidence-span distributions, builds flat records for every
split, and serialises them to disk. Model and training cells are excluded.

In [88]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR NOTEBOOK.
import kagglehub
seiftarek158_contract_nli_path = kagglehub.dataset_download('seiftarek158/contract-nli')
print('Data source import complete.')

Data source import complete.


### Data Engineering & Preparation

In [89]:
import os
import zipfile

extract_dir = seiftarek158_contract_nli_path
print(f'\nContents of {extract_dir}:')
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for fname in files:
        fpath = os.path.join(root, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f'{indent}  {fname}  ({size_kb:.1f} KB)')


Contents of /kaggle/input/datasets/seiftarek158/contract-nli:
contract-nli/
  contract-nli/
    TERMS  (3.6 KB)
    LICENSE  (18.2 KB)
    dev.json  (1155.4 KB)
    README.md  (5.4 KB)
    train.json  (7429.9 KB)
    test.json  (2207.2 KB)
    raw/
      1561604_0001193125-12-472390_d438799dex99d3.htm  (42.8 KB)
      1277141_0001193125-15-184193_d923647dex99d2.htm  (41.8 KB)
      clearcenter-rnda.pdf  (56.9 KB)
      nda-format-approved-by-legal-section.pdf  (221.9 KB)
      Non-Disclosure%20Agreement_3.pdf  (89.9 KB)
      917253_0000917253-00-000008_document_8.txt  (23.4 KB)
      Inaturals_NDA.pdf  (105.7 KB)
      confagree.pdf  (125.1 KB)
      828957_0000950109-98-001266_document_19.txt  (12.9 KB)
      Non-disclosure%20Agreement_1.pdf  (164.8 KB)
      59b1148ff6952b0001bdbedc_20170907_non%20disclosure%20agreement_expert.pdf  (175.7 KB)
      NDA-Instructions-Agreement-Attachment.pdf  (123.6 KB)
      25895_0000950134-07-023464_d51356exv10w1.htm  (28.1 KB)
      802724_000119

In [90]:
import json

DATA_DIR = seiftarek158_contract_nli_path

with open(f'{DATA_DIR}/contract-nli/train.json') as f:
    train_data = json.load(f)

print('Top-level keys:', list(train_data.keys()))
print('Number of documents:', len(train_data['documents']))
print('Number of hypotheses:', len(train_data['labels']))
print('\nHypothesis IDs and texts:')
for h_id, h_info in train_data['labels'].items():
    print(f'  {h_id}: {h_info}')

Top-level keys: ['documents', 'labels']
Number of documents: 423
Number of hypotheses: 17

Hypothesis IDs and texts:
  nda-11: {'short_description': 'No reverse engineering', 'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."}
  nda-16: {'short_description': 'Return of confidential information', 'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.'}
  nda-15: {'short_description': 'No licensing', 'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.'}
  nda-10: {'short_description': 'Confidentiality of Agreement', 'hypothesis': 'Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.'}
  nda-2: {'short_description': 'None-inclusion of non-technical information', 'hypothesis': 'Confidential Information shall only include technical information.'}
  nda-1: {'short_description'

In [91]:
doc = train_data['documents'][0]

print('Document ID:', doc['id'])
print('Contract length (chars):', len(doc['text']))
print('Number of spans:', len(doc['spans']))
print('\nFirst 3 spans (char ranges):')
for i, span in enumerate(doc['spans'][:3]):
    print(f"  span[{i}]: chars {span[0]}-{span[1]} -> {repr(doc['text'][span[0]:span[1]][:80])}")

print('\nAnnotation keys:', list(doc['annotation_sets'][0]['annotations'].keys()))

print('\nSample annotation (nda-1):')
ann = doc['annotation_sets'][0]['annotations']['nda-1']
print('  choice:', ann['choice'])
print('  span indices:', ann['spans'])
if ann['spans']:
    idx = ann['spans'][0]
    cs, ce = doc['spans'][idx]
    print(f"  resolved quote (span[{idx}], {cs}-{ce}): {repr(doc['text'][cs:ce][:120])}")

Document ID: 34
Contract length (chars): 8585
Number of spans: 65

First 3 spans (char ranges):
  span[0]: chars 0-44 -> 'NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT'
  span[1]: chars 45-132 -> 'This NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”) is made by and b'
  span[2]: chars 133-331 -> '(i) the Office of the United Nations High Commissioner for Refugees, having its '

Annotation keys: ['nda-11', 'nda-16', 'nda-15', 'nda-10', 'nda-2', 'nda-1', 'nda-19', 'nda-12', 'nda-20', 'nda-3', 'nda-18', 'nda-7', 'nda-17', 'nda-8', 'nda-13', 'nda-5', 'nda-4']

Sample annotation (nda-1):
  choice: Entailment
  span indices: [14]
  resolved quote (span[14], 1294-1683): '1. “Confidential Information”, whenever used in this Agreement, shall mean any data, document, specification and other i'


In [92]:
from collections import Counter, defaultdict

LABEL_MAP = {
    'Entailment':   'ENTAILED',
    'Contradiction': 'CONTRADICTED',
    'NotMentioned': 'NOT_MENTIONED',
}

overall_counter    = Counter()
hyp_counter        = defaultdict(Counter)
evidence_counts    = []
no_evidence_labels = Counter()

for doc in train_data['documents']:
    anns = doc['annotation_sets'][0]['annotations']
    for h_id, ann in anns.items():
        label = LABEL_MAP[ann['choice']]
        overall_counter[label] += 1
        hyp_counter[h_id][label] += 1
        evidence_counts.append(len(ann['spans']))
        if len(ann['spans']) == 0:
            no_evidence_labels[label] += 1

total = sum(overall_counter.values())
print('=== Overall Label Distribution ===')
for label, count in overall_counter.most_common():
    print(f'  {label:<20} {count:>5}  ({100*count/total:.1f}%)')

print(f'\n=== Evidence Span Coverage ===')
print(f'  Total (doc, hypothesis) pairs: {total}')
with_evidence = sum(1 for c in evidence_counts if c > 0)
print(f'  Pairs with >=1 evidence span:   {with_evidence} ({100*with_evidence/total:.1f}%)')
print(f'  Pairs with 0 evidence spans:    {total - with_evidence} ({100*(total-with_evidence)/total:.1f}%)')
print(f'\n  Of zero-evidence pairs, by label:')
for label, count in no_evidence_labels.most_common():
    print(f'    {label:<20} {count}')

print(f'\n=== Per-Hypothesis Label Distribution ===')
for h_id in sorted(hyp_counter.keys(), key=lambda x: int(x.split('-')[1])):
    counts = hyp_counter[h_id]
    h_info = train_data['labels'][h_id]
    desc   = h_info.get('short_description', h_id) if isinstance(h_info, dict) else h_id
    print(f'  {h_id} ({desc})')
    for label in ['ENTAILED', 'CONTRADICTED', 'NOT_MENTIONED']:
        c = counts.get(label, 0)
        print(f'    {label:<20} {c:>4}  ({100*c/sum(counts.values()):.1f}%)')

=== Overall Label Distribution ===
  ENTAILED              3530  (49.1%)
  NOT_MENTIONED         2820  (39.2%)
  CONTRADICTED           841  (11.7%)

=== Evidence Span Coverage ===
  Total (doc, hypothesis) pairs: 7191
  Pairs with >=1 evidence span:   4371 (60.8%)
  Pairs with 0 evidence spans:    2820 (39.2%)

  Of zero-evidence pairs, by label:
    NOT_MENTIONED        2820

=== Per-Hypothesis Label Distribution ===
  nda-1 (Explicit identification)
    ENTAILED               91  (21.5%)
    CONTRADICTED          112  (26.5%)
    NOT_MENTIONED         220  (52.0%)
  nda-2 (None-inclusion of non-technical information)
    ENTAILED               23  (5.4%)
    CONTRADICTED          309  (73.0%)
    NOT_MENTIONED          91  (21.5%)
  nda-3 (Inclusion of verbally conveyed information)
    ENTAILED              270  (63.8%)
    CONTRADICTED            4  (0.9%)
    NOT_MENTIONED         149  (35.2%)
  nda-4 (Limited use)
    ENTAILED              364  (86.1%)
    CONTRADICTED          

In [93]:
import statistics

lengths = [len(doc['text']) for doc in train_data['documents']]
print('=== Contract Length Stats (chars) ===')
print(f'  Min:    {min(lengths):>7}')
print(f'  Max:    {max(lengths):>7}')
print(f'  Mean:   {statistics.mean(lengths):>7.0f}')
print(f'  Median: {statistics.median(lengths):>7.0f}')
print(f'  Stdev:  {statistics.stdev(lengths):>7.0f}')

span_counts = [len(doc['spans']) for doc in train_data['documents']]
print(f'\n=== Spans per Document ===')
print(f'  Min:    {min(span_counts):>5}')
print(f'  Max:    {max(span_counts):>5}')
print(f'  Mean:   {statistics.mean(span_counts):>5.0f}')
print(f'  Median: {statistics.median(span_counts):>5.0f}')

# Load dev and test splits
with open(f'{DATA_DIR}/contract-nli/dev.json') as f:
    dev_data = json.load(f)
with open(f'{DATA_DIR}/contract-nli/test.json') as f:
    test_data = json.load(f)

print(f'\n=== Split Sizes ===')
print(f"  train: {len(train_data['documents'])} docs")
print(f"  dev:   {len(dev_data['documents'])} docs")
print(f"  test:  {len(test_data['documents'])} docs")
print(f"  total: {len(train_data['documents'])+len(dev_data['documents'])+len(test_data['documents'])} docs")

test_doc  = test_data['documents'][0]
first_ann = list(test_doc['annotation_sets'][0]['annotations'].values())[0]
print(f'\n=== Test Set Annotation Check ===')
print(f"  First annotation choice: {repr(first_ann['choice'])}")
print(f'  (Empty/null = blind test set, has labels = labeled test set)')

=== Contract Length Stats (chars) ===
  Min:       1481
  Max:      54571
  Mean:     11049
  Median:    9936
  Stdev:     6651

=== Spans per Document ===
  Min:       18
  Max:      354
  Mean:      78
  Median:    71

=== Split Sizes ===
  train: 423 docs
  dev:   61 docs
  test:  123 docs
  total: 607 docs

=== Test Set Annotation Check ===
  First annotation choice: 'NotMentioned'
  (Empty/null = blind test set, has labels = labeled test set)


In [94]:
# DATA_DIR_NLI points into the contract-nli/ subfolder
DATA_DIR_NLI = f'{DATA_DIR}/contract-nli'

LABEL_MAP = {
    'Entailment':   'ENTAILED',
    'Contradiction': 'CONTRADICTED',
    'NotMentioned': 'NOT_MENTIONED',
}

def get_gold_label(doc, hypothesis_id):
    ann = doc['annotation_sets'][0]['annotations'].get(hypothesis_id)
    if ann is None:
        return None
    return LABEL_MAP.get(ann['choice'], ann['choice'])

def get_evidence_spans(doc, hypothesis_id):
    """Return resolved evidence spans with char_start, char_end, quote."""
    ann = doc['annotation_sets'][0]['annotations'].get(hypothesis_id)
    if ann is None:
        return []
    result = []
    for span_idx in ann.get('spans', []):
        # doc['spans'] is a list of [char_start, char_end] pairs
        char_start, char_end = doc['spans'][span_idx]
        result.append({
            'span_index': span_idx,
            'char_start': char_start,
            'char_end':   char_end,
            'quote':      doc['text'][char_start:char_end],
        })
    return result

def build_all_records(data):
    """Flatten (document x hypothesis) into individual records."""
    hypotheses = data['labels']
    records = []
    for doc in data['documents']:
        for h_id in hypotheses:
            label = get_gold_label(doc, h_id)
            if label is not None:
                h_info     = hypotheses[h_id]
                hyp_text   = h_info['hypothesis']           if isinstance(h_info, dict) else h_info
                short_desc = h_info.get('short_description', h_id) if isinstance(h_info, dict) else h_id
                records.append({
                    'doc_id':            doc['id'],
                    'hypothesis_id':     h_id,
                    'hypothesis_text':   hyp_text,
                    'short_description': short_desc,
                    'gold_label':        label,
                    'evidence':          get_evidence_spans(doc, h_id),
                    'contract_text':     doc['text'],
                    'contract_length':   len(doc['text']),
                })
    return records

# Re-load all three splits from DATA_DIR_NLI
with open(f'{DATA_DIR_NLI}/train.json') as f:
    train_data = json.load(f)
with open(f'{DATA_DIR_NLI}/dev.json') as f:
    dev_data = json.load(f)
with open(f'{DATA_DIR_NLI}/test.json') as f:
    test_data = json.load(f)

train_records = build_all_records(train_data)
dev_records   = build_all_records(dev_data)
test_records  = build_all_records(test_data)

print(f'Train records: {len(train_records)}')
print(f'Dev records:   {len(dev_records)}')
print(f'Test records:  {len(test_records)}')
print(f'Total:         {len(train_records)+len(dev_records)+len(test_records)}')
print(f"\nSample record keys: {list(train_records[0].keys())}")
print(f"Sample label:       {train_records[0]['gold_label']}")
print(f"Sample hypothesis:  {train_records[0]['hypothesis_text']}")
print(f"Sample evidence count: {len(train_records[0]['evidence'])}")

Train records: 7191
Dev records:   1037
Test records:  2091
Total:         10319

Sample record keys: ['doc_id', 'hypothesis_id', 'hypothesis_text', 'short_description', 'gold_label', 'evidence', 'contract_text', 'contract_length']
Sample label:       NOT_MENTIONED
Sample hypothesis:  Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
Sample evidence count: 0


In [95]:
# Token budget analysis — rule of thumb: 1 token ~ 4 chars for legal English

def estimate_tokens(text):
    return len(text) // 4

hypothesis_avg_chars = sum(len(r['hypothesis_text']) for r in train_records) / len(train_records)
print(f'=== Token Budget Analysis ===')
print(f'Avg hypothesis length (chars): {hypothesis_avg_chars:.0f} (~{hypothesis_avg_chars//4:.0f} tokens)')

seen = set()
unique_contract_tokens = []
for r in train_records:
    if r['doc_id'] not in seen:
        seen.add(r['doc_id'])
        unique_contract_tokens.append(estimate_tokens(r['contract_text']))

print(f"\n=== Unique Contract Token Estimates (train, {len(unique_contract_tokens)} docs) ===")
print(f'  Min:    {min(unique_contract_tokens):>6}')
print(f'  Max:    {max(unique_contract_tokens):>6}')
print(f'  Mean:   {statistics.mean(unique_contract_tokens):>6.0f}')
print(f'  Median: {statistics.median(unique_contract_tokens):>6.0f}')
print(f'  Stdev:  {statistics.stdev(unique_contract_tokens):>6.0f}')

for limit in [1024, 2048, 4096, 8192]:
    fits = sum(1 for t in unique_contract_tokens if t <= limit)
    pct  = 100 * fits / len(unique_contract_tokens)
    print(f'  Contracts fitting in {limit:>5} tokens: {fits:>4} / {len(unique_contract_tokens)} ({pct:.1f}%)')

evidence_token_lengths = []
for r in train_records:
    for e in r['evidence']:
        evidence_token_lengths.append(estimate_tokens(e['quote']))

if evidence_token_lengths:
    print(f'\n=== Evidence Span Token Lengths ===')
    print(f'  Count:  {len(evidence_token_lengths)}')
    print(f'  Min:    {min(evidence_token_lengths)}')
    print(f'  Max:    {max(evidence_token_lengths)}')
    print(f'  Mean:   {statistics.mean(evidence_token_lengths):.0f}')
    print(f'  Median: {statistics.median(evidence_token_lengths):.0f}')

=== Token Budget Analysis ===
Avg hypothesis length (chars): 97 (~24 tokens)

=== Unique Contract Token Estimates (train, 423 docs) ===
  Min:       370
  Max:     13642
  Mean:     2762
  Median:   2484
  Stdev:    1663
  Contracts fitting in  1024 tokens:   40 / 423 (9.5%)
  Contracts fitting in  2048 tokens:  159 / 423 (37.6%)
  Contracts fitting in  4096 tokens:  354 / 423 (83.7%)
  Contracts fitting in  8192 tokens:  419 / 423 (99.1%)

=== Evidence Span Token Lengths ===
  Count:  8341
  Min:    0
  Max:    401
  Mean:   63
  Median: 50


In [96]:
# Compute relative position of each evidence span within its contract
positions = []

for doc in train_data['documents']:
    contract_len = len(doc['text'])
    anns = doc['annotation_sets'][0]['annotations']
    for h_id, ann in anns.items():
        label = LABEL_MAP[ann['choice']]
        for span_idx in ann.get('spans', []):
            cs, ce = doc['spans'][span_idx]
            mid = (cs + ce) / 2
            rel = mid / contract_len
            positions.append({'rel_pos': rel, 'h_id': h_id, 'label': label,
                              'char_start': cs, 'char_end': ce, 'contract_len': contract_len})

buckets = [0] * 10
for p in positions:
    bucket = min(int(p['rel_pos'] * 10), 9)
    buckets[bucket] += 1

total_spans = len(positions)
bucket_pcts = [round(100 * b / total_spans, 1) for b in buckets]

print('Decile distribution of evidence spans:')
for i, pct in enumerate(bucket_pcts):
    bar = '#' * int(pct)
    print(f'  {i*10:>2}-{(i+1)*10}%  {bar} {pct}%')

cumulative = 0
print('\nCumulative coverage by truncation point:')
for i, pct in enumerate(bucket_pcts):
    cumulative += pct
    print(f'  Keep first {(i+1)*10}% of contract -> covers {cumulative:.1f}% of evidence spans')

print('\nbucket_pcts =', bucket_pcts)

Decile distribution of evidence spans:
   0-10%  ##### 5.6%
  10-20%  ################## 18.2%
  20-30%  ################# 17.9%
  30-40%  ################ 16.3%
  40-50%  ############# 13.5%
  50-60%  ########### 11.6%
  60-70%  ####### 7.8%
  70-80%  ##### 5.1%
  80-90%  ## 2.3%
  90-100%  # 1.7%

Cumulative coverage by truncation point:
  Keep first 10% of contract -> covers 5.6% of evidence spans
  Keep first 20% of contract -> covers 23.8% of evidence spans
  Keep first 30% of contract -> covers 41.7% of evidence spans
  Keep first 40% of contract -> covers 58.0% of evidence spans
  Keep first 50% of contract -> covers 71.5% of evidence spans
  Keep first 60% of contract -> covers 83.1% of evidence spans
  Keep first 70% of contract -> covers 90.9% of evidence spans
  Keep first 80% of contract -> covers 96.0% of evidence spans
  Keep first 90% of contract -> covers 98.3% of evidence spans
  Keep first 100% of contract -> covers 100.0% of evidence spans

bucket_pcts = [5.6, 18.2, 

In [97]:
# Truncation limit: 1700 tokens * 4 chars/token = 6800 chars
CONTRACT_CHAR_LIMIT = 1700 * 4

def build_training_prompt(record, include_answer=True):
    """
    Training-format prompt for a single (doc, hypothesis) pair.
    include_answer=True  -> training format (input + output JSON)
    include_answer=False -> inference input only

    Truncation: skip first 10% of contract (only ~5.6% of evidence spans live
    there), then take CONTRACT_CHAR_LIMIT chars.
    """
    contract = record['contract_text']

    if len(contract) > CONTRACT_CHAR_LIMIT:
        offset        = int(len(contract) * 0.10)
        visible_start = offset
        visible_end   = offset + CONTRACT_CHAR_LIMIT
        contract      = contract[offset: offset + CONTRACT_CHAR_LIMIT] + '\n[TRUNCATED]'
    else:
        visible_start = 0
        visible_end   = len(contract)

    input_part = (
        'Classify the hypothesis based on the contract. '
        'Respond with ONLY valid JSON nothing else:\n'
        '{"label": "ENTAILED" | "CONTRADICTED" | "NOT_MENTIONED", "evidence": ["exact quote 1", "exact quote 2"]}\n'
        'If label is NOT_MENTIONED, evidence must be [].\n'
        'Evidence must be copied verbatim from the contract text, word for word. Do not paraphrase or invent.\n'
        f"Contract:\n{contract}\n\n"
        f"Hypothesis:\n{record['hypothesis_text']}\n\n"
    )

    if not include_answer:
        return input_part

    visible_evidence = [
        e['quote'].strip().replace('\n', ' ')
        for e in record['evidence']
        if e['char_start'] >= visible_start and e['char_end'] <= visible_end
    ]

    output_part = json.dumps({'label': record['gold_label'], 'evidence': visible_evidence})
    return input_part + output_part


# Quick sanity check on a few samples
for label_target in ['ENTAILED', 'CONTRADICTED', 'NOT_MENTIONED']:
    sample     = next(r for r in train_records if r['gold_label'] == label_target)
    prompt     = build_training_prompt(sample, include_answer=True)
    tokens_est = len(prompt) // 4
    print(f'=== {label_target} example ===')
    print(f'  Total prompt length: {len(prompt)} chars (~{tokens_est} tokens)')
    print(f'  Contract truncated:  {len(sample["contract_text"]) > CONTRACT_CHAR_LIMIT}')
    print(f'  TAIL (last 150 chars): {repr(prompt[-150:])}')
    print()

=== ENTAILED example ===
  Total prompt length: 7653 chars (~1913 tokens)
  Contract truncated:  True
  TAIL (last 150 chars): 'lationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or"]}'

=== CONTRADICTED example ===
  Total prompt length: 7564 chars (~1891 tokens)
  Contract truncated:  True
  TAIL (last 150 chars): ', software, procedures, products, services, development projects, and programmes contained in such Idea and/or its description and any conclusions."]}'

=== NOT_MENTIONED example ===
  Total prompt length: 7336 chars (~1834 tokens)
  Contract truncated:  True
  TAIL (last 150 chars): 'ing Party shall not reverse engineer any objects which embody Disclosing Party\'s Confidential Information.\n\n{"label": "NOT_MENTIONED", "evidence": []}'



In [98]:
from pathlib import Path

PREPROCESSED_DIR = Path('/kaggle/working/preprocessed')
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def serialize_split(records, split_name):
    output          = []
    truncated_count = 0

    for r in records:
        was_truncated = len(r['contract_text']) > CONTRACT_CHAR_LIMIT
        if was_truncated:
            truncated_count += 1
        output.append({
            'doc_id':                r['doc_id'],
            'hypothesis_id':         r['hypothesis_id'],
            'split':                 split_name,
            'gold_label':            r['gold_label'],
            'evidence':              r['evidence'],
            'prompt_train':          build_training_prompt(r, include_answer=True),
            'prompt_inference':      build_training_prompt(r, include_answer=False),
            'contract_length_chars': r['contract_length'],
            'was_truncated':         was_truncated,
            'hypothesis_text':       r['hypothesis_text'],
            'short_description':     r['short_description'],
        })

    path = PREPROCESSED_DIR / f'{split_name}.json'
    with open(path, 'w') as f:
        json.dump(output, f, indent=2)

    print(f'=== {split_name} ===')
    print(f'  Records written:  {len(output)}')
    print(f'  Truncated:        {truncated_count} ({100*truncated_count/len(output):.1f}%)')
    print(f'  Saved to:         {path}')
    print(f'  File size:        {os.path.getsize(path)/1024/1024:.1f} MB')
    return output

train_out = serialize_split(train_records, 'train')
dev_out   = serialize_split(dev_records,   'dev')
test_out  = serialize_split(test_records,  'test')

# Sanity check
import random
sample = random.choice(train_out)
print(f'\n=== Round-trip Sanity Check ===')
print(f"  doc_id:         {sample['doc_id']}")
print(f"  hypothesis_id:  {sample['hypothesis_id']}")
print(f"  gold_label:     {sample['gold_label']}")
print(f"  was_truncated:  {sample['was_truncated']}")
print(f"  train prompt ends with: {repr(sample['prompt_train'][-60:])}")
print(f"  inference prompt ends with: {repr(sample['prompt_inference'][-60:])}")

=== train ===
  Records written:  7191
  Truncated:        5066 (70.4%)
  Saved to:         /kaggle/working/preprocessed/train.json
  File size:        101.1 MB
=== dev ===
  Records written:  1037
  Truncated:        816 (78.7%)
  Saved to:         /kaggle/working/preprocessed/dev.json
  File size:        15.0 MB
=== test ===
  Records written:  2091
  Truncated:        1479 (70.7%)
  Saved to:         /kaggle/working/preprocessed/test.json
  File size:        29.2 MB

=== Round-trip Sanity Check ===
  doc_id:         344
  hypothesis_id:  nda-8
  gold_label:     ENTAILED
  was_truncated:  True
  train prompt ends with: 'r waive compliance with the provisions of this Agreement."]}'
  inference prompt ends with: 'judicial process to disclose any Confidential Information.\n\n'


## 2. Imports and Configuration (Inference)

In [99]:
import os
# Restrict to GPU 0 only. Must be set before torch/unsloth import.
# Prevents unsloth from pre-allocating rope-embedding cache on cuda:1,
# which caused AcceleratorError during model loading after threading crash.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel
import re
import time
import datetime
import warnings
import csv
from typing import Any

import torch
from transformers import AutoTokenizer
from peft import PeftModel


warnings.filterwarnings('ignore')
print('Libraries loaded.')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPUs visible: {torch.cuda.device_count()}')


Libraries loaded.
CUDA available: True
GPUs visible: 1


In [100]:
import os, zipfile
from pathlib import Path

BASE_MODEL_ID = "unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit"

# ── MANUAL OVERRIDE ────────────────────────────────────────────────────────
# Set this to the folder containing adapter_config.json, or leave None.
ADAPTER_PATH_OVERRIDE = "/kaggle/input/models/harridy/qwen/tensorflow2/default/1/kaggle/working/qwen3-4B-nli-lora-adapter"

# ── Auto-detect ─────────────────────────────────────────────────────────────
if ADAPTER_PATH_OVERRIDE:
    ADAPTER_PATH = ADAPTER_PATH_OVERRIDE
else:
    # Extract zip if uploaded
    for zp in ["/content/qwen-nli-lora-adapter (1).zip",
               "/content/qwen-nli-lora-adapter.zip",
               "/content/qwen3-4B-nli-lora-adapter.zip"]:
        if os.path.isfile(zp):
            print(f"Extracting {zp}...")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(os.path.dirname(zp))
            print("Done.")
            break

    ADAPTER_PATH = next(
        (p for p in [
            "/kaggle/working/qwen3-4B-nli-lora-adapter",
            "/content/qwen3-4B-nli-lora-adapter",
            "/content/kaggle/working/qwen3-4B-nli-lora-adapter",
            "/content/drive/MyDrive/qwen3-4B-nli-lora-adapter",
        ] if os.path.isfile(os.path.join(p, "adapter_config.json"))),
        None
    )

if ADAPTER_PATH is None:
    msg = (
        "Adapter not found. Options:"
        "  A) Upload qwen-nli-lora-adapter (1).zip to /content/ via Files panel, re-run."
        "  B) Mount Drive: from google.colab import drive; drive.mount('/content/drive')"
        "     then set ADAPTER_PATH_OVERRIDE above."
        "  C) Set ADAPTER_PATH_OVERRIDE manually at the top of this cell."
    )
    raise FileNotFoundError(msg)

MAX_NEW_TOKENS = 2048
BATCH_SIZE     = 1      # hypotheses per generate call — reduce if OOM
TEMPERATURE    = 0.3   
MAX_RETRIES    = 3

OUTPUT_DIR = (
    Path("/kaggle/working/outputs") if os.path.isdir("/kaggle/working")
    else Path("/content/outputs")
)
RUNTRACE_DIR     = OUTPUT_DIR / "runtraces"
EVAL_CSV_PATH    = OUTPUT_DIR / "evaluation_metrics.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "predictions.json"
OUTPUT_DIR.mkdir(exist_ok=True)
RUNTRACE_DIR.mkdir(exist_ok=True)

print(f"Adapter : {ADAPTER_PATH}")
print(f"Outputs : {OUTPUT_DIR}")

Adapter : /kaggle/input/models/harridy/qwen/tensorflow2/default/1/kaggle/working/qwen3-4B-nli-lora-adapter
Outputs : /kaggle/working/outputs


## 3. Load Model and Tokenizer

In [101]:
print("Loading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = 8192,
    dtype          = None,
    load_in_4bit   = True,
)

# ── Attach LoRA adapter manually ──────────────────────────────────────────────
# PeftModel.from_pretrained rejects local paths with >1 slash via HF Hub
# validation. Workaround: read config directly, build LoraConfig, load weights.
import json as _json
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file as _load_safetensors

print("Reading adapter config...")
with open(f"{ADAPTER_PATH}/adapter_config.json") as _f:
    _cfg = _json.load(_f)

lora_config = LoraConfig(
    r              = _cfg["r"],
    lora_alpha     = _cfg["lora_alpha"],
    lora_dropout   = _cfg["lora_dropout"],
    target_modules = _cfg["target_modules"],
    bias           = _cfg.get("bias", "none"),
    task_type      = "CAUSAL_LM",
    inference_mode = True,
)

print("Wrapping model with LoRA...")
model = get_peft_model(model, lora_config)

print("Loading adapter weights...")
_weights = _load_safetensors(f"{ADAPTER_PATH}/adapter_model.safetensors", device="cpu")
set_peft_model_state_dict(model, _weights)

# Enable unsloth inference optimizations (2x faster generation)
model = FastLanguageModel.for_inference(model)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
print("Model ready.")


Loading base model...
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Reading adapter config...
Wrapping model with LoRA...
Loading adapter weights...
Model ready.


In [102]:
# Load a second copy of the model on GPU 1 for parallel inference.
# Each GPU handles half the contracts simultaneously -> ~2x speedup.
import os
print(f"GPUs available: {torch.cuda.device_count()}")

if torch.cuda.device_count() >= 2:
    print("Loading second model on cuda:1...")
    model_gpu1, _ = FastLanguageModel.from_pretrained(
        model_name     = BASE_MODEL_ID,
        max_seq_length = 8192,
        dtype          = None,
        load_in_4bit   = True,
        device_map     = "cuda:1",
    )

    import json as _json2
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
    from safetensors.torch import load_file as _load_sf2

    with open(f"{ADAPTER_PATH}/adapter_config.json") as _f:
        _cfg2 = _json2.load(_f)

    _lora2 = LoraConfig(
        r=_cfg2["r"], lora_alpha=_cfg2["lora_alpha"],
        lora_dropout=_cfg2["lora_dropout"], target_modules=_cfg2["target_modules"],
        bias=_cfg2.get("bias", "none"), task_type="CAUSAL_LM", inference_mode=True,
    )
    model_gpu1 = get_peft_model(model_gpu1, _lora2)
    _w2 = _load_sf2(f"{ADAPTER_PATH}/adapter_model.safetensors", device="cuda:1")
    set_peft_model_state_dict(model_gpu1, _w2)
    model_gpu1 = FastLanguageModel.for_inference(model_gpu1)
    print("GPU 1 model ready.")
else:
    model_gpu1 = None
    print("Only 1 GPU found — parallel inference disabled, will use GPU 0 only.")


GPUs available: 1
Only 1 GPU found — parallel inference disabled, will use GPU 0 only.


## 4. Define the 17 Hypotheses (H01–H17)

Pulled directly from `train_data['labels']` (loaded in Section 1).
The key path is `train_data['labels']['nda-1']['hypothesis']`.
This guarantees the texts match the dataset — no hardcoded drift.


In [103]:
def _extract_hyp_text(meta) -> str:
    if isinstance(meta, dict):
        for key in ('hypothesis', 'long_description', 'short_description'):
            if key in meta and meta[key]:
                return meta[key]
        return str(meta)
    return str(meta)

HYPOTHESES: dict[str, str] = {
    hid: _extract_hyp_text(meta)
    for hid, meta in sorted(
        train_data['labels'].items(),
        key=lambda kv: int(kv[0].split('-')[1])
    )
}

HYP_DISPLAY = {hid: f'H{i+1:02d}' for i, hid in enumerate(HYPOTHESES)}

print(f'Loaded {len(HYPOTHESES)} hypotheses from dataset.')
sample_val = list(train_data['labels'].values())[0]
print(f'Label metadata keys: {list(sample_val.keys()) if isinstance(sample_val, dict) else "string"}')
for hid, text in HYPOTHESES.items():
    print(f'  {HYP_DISPLAY[hid]} ({hid}): {text[:80]}')

Loaded 17 hypotheses from dataset.
Label metadata keys: ['short_description', 'hypothesis']
  H01 (nda-1): All Confidential Information shall be expressly identified by the Disclosing Par
  H02 (nda-2): Confidential Information shall only include technical information.
  H03 (nda-3): Confidential Information may include verbally conveyed information.
  H04 (nda-4): Receiving Party shall not use any Confidential Information for any purpose other
  H05 (nda-5): Receiving Party may share some Confidential Information with some of Receiving P
  H06 (nda-7): Receiving Party may share some Confidential Information with some third-parties 
  H07 (nda-8): Receiving Party shall notify Disclosing Party in case Receiving Party is require
  H08 (nda-10): Receiving Party shall not disclose the fact that Agreement was agreed or negotia
  H09 (nda-11): Receiving Party shall not reverse engineer any objects which embody Disclosing P
  H10 (nda-12): Receiving Party may independently develop information

## 5. Inference Utilities

In [104]:
# Qwen3 chat-format prompt builder
SYSTEM_PROMPT = """You are a legal Natural Language Inference (NLI) expert.
Given a contract text and a hypothesis, determine whether the hypothesis is
ENTAILED by, CONTRADICTED by, or NOT_MENTIONED in the contract.

Output ONLY valid JSON (no markdown fences, no extra text):
{
  "hypothesis_id": "<string>",
  "label": "<ENTAILED|CONTRADICTED|NOT_MENTIONED>",
  "confidence": <float 0.0-1.0>,
  "evidence": [
    {
      "char_start": <int>,
      "char_end": <int>,
      "quote": "<exact substring of contract>",
      "relevance_score": <float 0.0-1.0>
    }
  ]
}

Rules:
1. ENTAILED or CONTRADICTED -> at least one evidence span required.
2. NOT_MENTIONED -> evidence list may be empty.
3. char_start/char_end are 0-based character indices into the CONTRACT below.
4. quote must equal contract[char_start:char_end] exactly.
5. confidence is your certainty in the label (1.0 = certain, 0.0 = no evidence either way).
6. relevance_score per evidence span: how strongly does this quote support your label (1.0 = very strong, 0.0 = weak)."""


def build_prompt(contract_text: str, hyp_id: str, hyp_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"CONTRACT:{contract_text}"
            f"HYPOTHESIS ({HYP_DISPLAY[hyp_id]} / {hyp_id}): {hyp_text}"
        )},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

In [105]:
import re
from json_repair import repair_json

_THINK_RE      = re.compile(r"<think>[\s\S]*?</think>", re.DOTALL)
_FENCE_RE      = re.compile(r"```(?:json)?\s*")
_JSON_BLOCK_RE = re.compile(r"\{[\s\S]*\}", re.DOTALL)


def extract_json(raw_text: str) -> dict | None:
    # 1. Strip Qwen3 thinking block <think>...</think>
    text = _THINK_RE.sub("", raw_text).strip()
    # 2. Strip markdown code fences
    text = _FENCE_RE.sub("", text).strip()
    # 3. Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # 4. Find first {...} block
    match = _JSON_BLOCK_RE.search(text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            # 5. Repair malformed JSON (unescaped newlines, truncated strings, etc.)
            try:
                repaired = repair_json(match.group(), return_objects=True)
                if isinstance(repaired, dict):
                    return repaired
            except Exception:
                pass
    # 6. Last resort: repair the whole text
    try:
        repaired = repair_json(text, return_objects=True)
        if isinstance(repaired, dict):
            return repaired
    except Exception:
        pass
    return None


VALID_LABELS = {"ENTAILED", "CONTRADICTED", "NOT_MENTIONED"}


def validate_prediction(pred: dict, hyp_id: str) -> tuple[bool, list[str]]:
    errors: list[str] = []
    for key in ("hypothesis_id", "label", "evidence"):
        if key not in pred:
            errors.append(f"Missing key: '{key}'")
    if errors:
        return False, errors

    label    = pred["label"]
    evidence = pred["evidence"]

    if label not in VALID_LABELS:
        errors.append(f"Invalid label '{label}'")
    if not isinstance(evidence, list):
        errors.append("'evidence' must be a list")
        return False, errors
    if label in ("ENTAILED", "CONTRADICTED") and len(evidence) == 0:
        errors.append(f"Label is '{label}' but no evidence spans (Groundedness rule)")

    for i, span in enumerate(evidence):
        for field in ("char_start", "char_end", "quote"):
            if field not in span:
                errors.append(f"Span[{i}] missing '{field}'")
        # relevance_score is optional; if present must be float in [0,1]
        if "relevance_score" in span:
            if not isinstance(span["relevance_score"], (int, float)):
                errors.append(f"Span[{i}] relevance_score must be numeric")
            elif not (0.0 <= span["relevance_score"] <= 1.0):
                errors.append(f"Span[{i}] relevance_score must be in [0.0, 1.0]")
        if "char_start" in span and "char_end" in span:
            if not (isinstance(span["char_start"], int) and isinstance(span["char_end"], int)):
                errors.append(f"Span[{i}] char_start/char_end must be integers")
            elif span["char_start"] >= span["char_end"]:
                errors.append(f"Span[{i}] char_start >= char_end")

    return len(errors) == 0, errors


In [106]:
@torch.inference_mode()
def run_inference_batch(
    contract_text: str,
    hypotheses: dict,
    _model=None,
    _device: str = "cuda:0",
) -> list[dict]:
    """
    Process hypotheses in mini-batches of BATCH_SIZE on the given device.
    Pass _model and _device to run on a specific GPU.
    """
    if _model is None:
        _model = model   # default to GPU 0 model

    hyp_ids = list(hypotheses.keys())
    prompts = [build_prompt(contract_text, hid, hypotheses[hid]) for hid in hyp_ids]
    all_raw: dict[str, str] = {}

    for batch_start in range(0, len(hyp_ids), BATCH_SIZE):
        b_ids     = hyp_ids[batch_start: batch_start + BATCH_SIZE]
        b_prompts = prompts[batch_start: batch_start + BATCH_SIZE]

        enc = tokenizer(
            b_prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=7680,
        )
        enc       = {k: v.to(_device) for k, v in enc.items()}
        input_len = enc["input_ids"].shape[1]

        out_ids = _model.generate(
            **enc,
            max_new_tokens = MAX_NEW_TOKENS,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

        for i, hid in enumerate(b_ids):
            all_raw[hid] = tokenizer.decode(
                out_ids[i][input_len:], skip_special_tokens=True
            ).strip()

        del enc, out_ids
        torch.cuda.empty_cache()

    results = []
    for hyp_id in hyp_ids:
        raw_text = all_raw[hyp_id]
        parsed   = extract_json(raw_text)

        if parsed is None:
            print(f"    [{hyp_id}] parse failed, retrying individually on {_device}...")
            idx   = hyp_ids.index(hyp_id)
            s_enc = tokenizer(prompts[idx], return_tensors="pt",
                              truncation=True, max_length=7680)
            s_enc = {k: v.to(_device) for k, v in s_enc.items()}
            for attempt in range(MAX_RETRIES):
                s_out = _model.generate(
                    **s_enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
                raw_text = tokenizer.decode(
                    s_out[0][s_enc["input_ids"].shape[1]:], skip_special_tokens=True
                ).strip()
                parsed = extract_json(raw_text)
                if parsed is not None:
                    break
                print(f"      Attempt {attempt+1}/{MAX_RETRIES} failed.")
            del s_enc

        if parsed is None:
            print(f"    *** FLAGGED: {hyp_id} all retries exhausted")
            results.append({
                "prediction": {"hypothesis_id": hyp_id, "label": "NOT_MENTIONED", "evidence": []},
                "raw_output": raw_text, "is_valid": False,
                "errors": ["JSON extraction failed after retries"],
                "flagged": True, "attempts": MAX_RETRIES + 1,
            })
            continue

        parsed.setdefault("hypothesis_id", hyp_id)
        parsed.setdefault("evidence", [])
        is_valid, errors = validate_prediction(parsed, hyp_id)
        if not is_valid:
            print(f"    [FLAGGED] {hyp_id} validation: {errors}")
        results.append({
            "prediction": parsed, "raw_output": raw_text,
            "is_valid": is_valid, "errors": errors,
            "flagged": not is_valid, "attempts": 1,
        })

    return results


## Step 5: Inference Pipeline

In [107]:
import time, json
from pathlib import Path

# ── Checkpoint / Resume ────────────────────────────────────────────────────
# After each contract, progress is saved atomically to checkpoint.json.
# Re-run this cell after a crash and it will skip already-finished contracts.
CHECKPOINT_FILE = Path(OUTPUT_DIR) / "checkpoint.json"

def _load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        try:
            with open(CHECKPOINT_FILE) as f:
                ckpt = json.load(f)
            # Normalise IDs to str (old checkpoints may have saved raw integers)
            ckpt["completed"] = [str(x) for x in ckpt.get("completed", [])]
            ckpt["results"]   = {str(k): v for k, v in ckpt.get("results", {}).items()}
            print(f"Checkpoint found: {len(ckpt['completed'])} contracts already done.")
            return ckpt
        except Exception as e:
            print(f"Warning: checkpoint corrupt ({e}), starting fresh.")
    return {"completed": [], "results": {}}

def _save_checkpoint(ckpt: dict):
    """Atomic write: write to .tmp then rename so a crash mid-write cannot corrupt the file."""
    tmp = CHECKPOINT_FILE.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(ckpt, f)
    tmp.replace(CHECKPOINT_FILE)   # atomic on POSIX; best-effort on Windows

# ── Setup ──────────────────────────────────────────────────────────────────
eval_documents = test_data["documents"]

# NOTE: dual-GPU Python-threading was removed because unsloth\'s rope-embedding
# CUDA kernels are NOT thread-safe (torch.AcceleratorError on concurrent calls).
# Sequential single-GPU inference + checkpoint/resume is used instead.
_infer_model  = model
_infer_device = "cuda:0"

# Restore completed work from checkpoint
ckpt          = _load_checkpoint()
completed_ids = set(ckpt["completed"])

# Pre-allocate result slots; fill already-finished ones from checkpoint
all_contract_results = [None] * len(eval_documents)
for i, doc in enumerate(eval_documents):
    cid = str(doc.get("id", i))
    if cid in completed_ids:
        all_contract_results[i] = ckpt["results"][cid]

n_done = len(completed_ids)
n_todo = len(eval_documents) - n_done
print(f"Contracts to process: {n_todo}  |  already done (skipping): {n_done}")

# ── Main inference loop ────────────────────────────────────────────────────
for idx, doc in enumerate(eval_documents):
    contract_id   = str(doc.get("id", idx))

    if contract_id in completed_ids:
        print(f"[skip] {contract_id}")
        continue

    full_text     = doc["text"]
    raw_spans     = doc["spans"]
    gold_anns     = doc["annotation_sets"][0]["annotations"]

    contract_text = full_text

    print(f"[{idx+1}/{len(eval_documents)}] {contract_id} ({len(contract_text)} chars)")
    t0 = time.perf_counter()

    batch_results = run_inference_batch(contract_text, HYPOTHESES, _infer_model, _infer_device)

    hyp_results = []
    for result, (hyp_id, _) in zip(batch_results, HYPOTHESES.items()):
        gold_entry     = gold_anns.get(hyp_id, {})
        gold_label_raw = gold_entry.get("choice", "NotMentioned")
        gold_label     = LABEL_MAP.get(gold_label_raw, gold_label_raw.upper())
        gold_idx       = gold_entry.get("spans", [])
        gold_spans     = [
            {"span_index": i, "char_start": raw_spans[i][0], "char_end": raw_spans[i][1]}
            for i in gold_idx if i < len(raw_spans)
        ]
        result["hyp_id"]              = hyp_id
        result["hyp_display"]         = HYP_DISPLAY[hyp_id]
        result["gold_label"]          = gold_label
        result["gold_spans_resolved"] = gold_spans
        result["raw_spans"]           = raw_spans

        pred = result["prediction"].get("label", "NOT_MENTIONED")
        flag = " [FLAGGED]" if result["flagged"] else ""
        print(f"  {HYP_DISPLAY[hyp_id]} ({hyp_id}): {pred} (gold={gold_label}){flag}")
        hyp_results.append(result)

    latency_ms      = (time.perf_counter() - t0) * 1000
    contract_record = {
        "contract_id":        contract_id,
        "latency_ms":         latency_ms,
        "hypothesis_results": hyp_results,
    }
    all_contract_results[idx] = contract_record

    # ── Checkpoint: save immediately after every contract ──────────────────
    ckpt["completed"].append(contract_id)
    ckpt["results"][contract_id] = contract_record
    _save_checkpoint(ckpt)
    print(f"  => Latency: {latency_ms:.0f} ms  [checkpoint saved]")

print("\nInference complete.")


Checkpoint found: 123 contracts already done.
Contracts to process: 0  |  already done (skipping): 123
[skip] 1
[skip] 2
[skip] 4
[skip] 5
[skip] 6
[skip] 8
[skip] 11
[skip] 18
[skip] 21
[skip] 22
[skip] 23
[skip] 24
[skip] 25
[skip] 26
[skip] 30
[skip] 31
[skip] 36
[skip] 38
[skip] 40
[skip] 41
[skip] 42
[skip] 43
[skip] 46
[skip] 47
[skip] 48
[skip] 49
[skip] 50
[skip] 51
[skip] 52
[skip] 53
[skip] 55
[skip] 57
[skip] 58
[skip] 59
[skip] 60
[skip] 62
[skip] 65
[skip] 66
[skip] 67
[skip] 68
[skip] 74
[skip] 76
[skip] 77
[skip] 78
[skip] 80
[skip] 83
[skip] 84
[skip] 85
[skip] 90
[skip] 97
[skip] 99
[skip] 117
[skip] 127
[skip] 129
[skip] 138
[skip] 157
[skip] 165
[skip] 181
[skip] 190
[skip] 212
[skip] 213
[skip] 218
[skip] 225
[skip] 227
[skip] 228
[skip] 289
[skip] 293
[skip] 296
[skip] 301
[skip] 307
[skip] 310
[skip] 321
[skip] 336
[skip] 354
[skip] 357
[skip] 361
[skip] 387
[skip] 389
[skip] 392
[skip] 395
[skip] 400
[skip] 401
[skip] 408
[skip] 412
[skip] 413
[skip] 429
[skip] 4

## Step 6: Evidence Span Validation

In [108]:
def validate_evidence_spans(
    contract_text: str,
    raw_spans: list,          # list of [char_start, char_end] from doc['spans']
    predicted_evidence: list[dict],
    hyp_id: str,
    contract_id: str,
) -> dict[str, Any]:
    """
    Step 6 - Evidence Span Validation.

    A. Quote-integrity: span['quote'] == contract_text[char_start:char_end]
    B. Canonical cross-reference: predicted [cs, ce) overlaps a canonical span
       from doc['spans'] (the dataset's [char_start, char_end] pairs).
    """
    canonical_intervals: list[tuple[int, int]] = [
        (int(sp[0]), int(sp[1])) for sp in raw_spans
    ]

    def overlaps(a_start: int, a_end: int) -> bool:
        return any(a_start < e and a_end > s for s, e in canonical_intervals)

    span_reports: list[dict] = []
    all_quotes_valid    = True
    all_canonical_match = True

    for i, span in enumerate(predicted_evidence):
        cs    = span.get('char_start')
        ce    = span.get('char_end')
        quote = span.get('quote', '')
        report: dict[str, Any] = {'span_index': i, 'char_start': cs, 'char_end': ce}

        # A. Quote-integrity check
        if cs is None or ce is None:
            report['quote_valid'] = False
            report['quote_error'] = 'Missing char_start or char_end'
            all_quotes_valid = False
        elif ce > len(contract_text):
            report['quote_valid'] = False
            report['quote_error'] = f'char_end ({ce}) exceeds contract length ({len(contract_text)})'
            all_quotes_valid = False
        else:
            actual = contract_text[cs:ce]
            if actual == quote:
                report['quote_valid'] = True
                report['quote_error'] = None
            else:
                report['quote_valid'] = False
                report['quote_error'] = (
                    f'MISMATCH - predicted: {repr(quote[:80])} | '
                    f'actual: {repr(actual[:80])}'
                )
                all_quotes_valid = False

        # B. Canonical cross-reference
        if cs is not None and ce is not None and canonical_intervals:
            matched = overlaps(cs, ce)
            report['canonical_match']   = matched
            report['canonical_warning'] = None if matched else f'Span [{cs}:{ce}] does not overlap any canonical span'
            if not matched:
                all_canonical_match = False
        else:
            report['canonical_match']   = None
            report['canonical_warning'] = None

        span_reports.append(report)

    return {
        'contract_id':         contract_id,
        'hyp_id':              hyp_id,
        'all_quotes_valid':    all_quotes_valid,
        'has_canonical_match': all_canonical_match,
        'span_reports':        span_reports,
    }

In [109]:
# Quick lookup: contract_id -> doc
doc_lookup = {str(doc.get('id', i)): doc for i, doc in enumerate(eval_documents)}

print('Running evidence span validation (Step 6)...')

for contract_record in all_contract_results:
    contract_id   = str(contract_record['contract_id'])
    doc           = doc_lookup.get(contract_id)
    if doc is None:
        print(f"  WARNING: Could not find doc '{contract_id}' in eval_documents.")
        continue
    full_text = doc['text']
    raw_spans = doc['spans']   # list of [char_start, char_end] pairs

    contract_text = full_text

    for hyp_result in contract_record['hypothesis_results']:
        hyp_id             = hyp_result['hyp_id']
        predicted_evidence = hyp_result['prediction'].get('evidence', [])

        validation_report = validate_evidence_spans(
            contract_text      = contract_text,
            raw_spans          = raw_spans,
            predicted_evidence = predicted_evidence,
            hyp_id             = hyp_id,
            contract_id        = contract_id,
        )
        hyp_result['evidence_validation'] = validation_report

        for sr in validation_report['span_reports']:
            if not sr['quote_valid']:
                print(f"  [QUOTE MISMATCH] {contract_id} / {hyp_id} span[{sr['span_index']}]: {sr['quote_error']}")
            if sr.get('canonical_warning'):
                print(f"  [CANONICAL MISS] {contract_id} / {hyp_id}: {sr['canonical_warning']}")

print('Evidence span validation complete.')


Running evidence span validation (Step 6)...
  [QUOTE MISMATCH] 1 / nda-1 span[0]: MISMATCH - predicted: 'a. the documents described in the Whereas clause above;' | actual: 'of those agreements are '
  [QUOTE MISMATCH] 1 / nda-1 span[1]: MISMATCH - predicted: 'b. Critical Infrastructure Information (CII) or Bulk Electric System Information' | actual: 'ncorporated herein.\nJEA '
  [QUOTE MISMATCH] 1 / nda-1 span[2]: MISMATCH - predicted: 'c. Protected Health Information in both physical and electronic form (PHI and eP' | actual: ' Florida State Sunshine '
  [QUOTE MISMATCH] 1 / nda-1 span[3]: MISMATCH - predicted: 'd. Personal Identifiable Information (PII)' | actual: 'pplication - JEA is a pu'
  [QUOTE MISMATCH] 1 / nda-1 span[4]: MISMATCH - predicted: 'e. any protected, non-public information concerning the design or operation of p' | actual: 'licly owned utility and '
  [QUOTE MISMATCH] 1 / nda-1 span[5]: MISMATCH - predicted: 'f. any information that could be used to compromise or e

## Step 7: Playbook YAML Loader & Deterministic Mapping Engine

In [110]:
import yaml, hashlib, uuid

# ── Auto-detect playbook.yaml ──────────────────────────────────────────────
PLAYBOOK_PATH_OVERRIDE = "/kaggle/input/datasets/harridy/extras/playbook.yaml"   # e.g. "/kaggle/working/playbook.yaml"

PLAYBOOK_PATH = PLAYBOOK_PATH_OVERRIDE or next(
    (p for p in [
        "/kaggle/working/playbook.yaml",
        "/kaggle/input/contractnli-playbook/playbook.yaml",
        "/content/playbook.yaml",
        "/content/drive/MyDrive/playbook.yaml",
    ] if os.path.isfile(p)),
    None,
)

if PLAYBOOK_PATH is None:
    raise FileNotFoundError(
        "playbook.yaml not found. Upload it to /kaggle/working/ and re-run."
    )

with open(PLAYBOOK_PATH, "r") as _f:
    _pb_raw = _f.read()

playbook_yaml         = yaml.safe_load(_pb_raw)
PLAYBOOK_RULESET_HASH = hashlib.sha256(_pb_raw.encode()).hexdigest()

print(f"Playbook : {playbook_yaml['playbook_id']}  v{playbook_yaml['version']}")
print(f"SHA-256  : {PLAYBOOK_RULESET_HASH}")

# ── Build lookup H01..H17 → check config ──────────────────────────────────
_pb_checks  = {c["hypothesis_id"]: c for c in playbook_yaml["checks"]}
_pb_globals = playbook_yaml["global_defaults"]
_lbl_status  = _pb_globals["label_to_status"]
_lbl_default = _pb_globals["label_to_default_decision"]

def apply_playbook(hyp_display: str, label: str) -> dict:
    """Deterministic (H01..H17, label) → {severity, action, criticality, status, rationale}."""
    check       = _pb_checks.get(hyp_display, {})
    criticality = check.get("criticality", "P2")
    status      = _lbl_status.get(label, "missing")

    overrides = check.get("overrides", {})
    if label in overrides:
        severity = overrides[label]["severity"]
        action   = overrides[label]["action"]
    else:
        severity = _lbl_default[label]["severity"]
        action   = _lbl_default[label]["action"]

    title = check.get("title", hyp_display)
    tmpl  = check.get("rationale_templates", {}).get(
        status, "{HYPOTHESIS_TITLE}: Severity {SEVERITY}; action {ACTION}."
    )
    rationale = (tmpl
        .replace("{HYPOTHESIS_TITLE}", title)
        .replace("{STATUS}", status)
        .replace("{SEVERITY}", severity)
        .replace("{ACTION}", action)
        .replace("{EVIDENCE_SUMMARY}", "")
        .replace("{TOP_CITATION}", "")
        .replace("{CONFIDENCE}", "")
        .strip()
    )

    return {
        "severity":    severity,
        "action":      action,
        "criticality": criticality,
        "status":      status,
        "rationale":   rationale,
        "rule_ids":    [f"{hyp_display}_{label}"],
    }

# ── Smoke test ─────────────────────────────────────────────────────────────
for _hid, _lbl in [("H04", "NOT_MENTIONED"), ("H01", "ENTAILED"), ("H06", "CONTRADICTED")]:
    _r = apply_playbook(_hid, _lbl)
    print(f"  {_hid}/{_lbl:14s} -> severity={_r['severity']:6s} action={_r['action']}")


Playbook : contractnli_nda_minimal  v1.1
SHA-256  : 522434a49314eebf038a96bdf24de823eb9f62ab090c1265528bb71cde3af51f
  H04/NOT_MENTIONED  -> severity=HIGH   action=CLARIFY
  H01/ENTAILED       -> severity=LOW    action=ACCEPT
  H06/CONTRADICTED   -> severity=HIGH   action=NEGOTIATE


## 7. Compute Evaluation Metrics

In [111]:
n_total = n_correct = n_compliant = n_quote_valid = 0
total_latency_ms = 0.0

for contract_record in all_contract_results:
    total_latency_ms += contract_record['latency_ms']
    for hyp_result in contract_record['hypothesis_results']:
        n_total    += 1
        pred_label = hyp_result['prediction'].get('label', 'NOT_MENTIONED')
        gold_label = hyp_result['gold_label']
        evidence   = hyp_result['prediction'].get('evidence', [])
        ev_val     = hyp_result.get('evidence_validation', {})

        if pred_label == gold_label:
            n_correct += 1
        if pred_label in ('ENTAILED', 'CONTRADICTED'):
            if len(evidence) >= 1:
                n_compliant += 1
        else:
            n_compliant += 1
        if ev_val.get('all_quotes_valid', True):
            n_quote_valid += 1

n_contracts          = len(all_contract_results)
label_accuracy       = n_correct     / n_total    if n_total    > 0 else 0.0
groundedness         = n_compliant   / n_total    if n_total    > 0 else 0.0
quote_integrity_rate = n_quote_valid / n_total    if n_total    > 0 else 0.0
avg_latency_ms       = total_latency_ms / n_contracts if n_contracts > 0 else 0.0

print('=' * 50)
print(f'  Contracts evaluated : {n_contracts}')
print(f'  Hypothesis instances: {n_total}')
print(f'  Label Accuracy      : {label_accuracy:.4f}  ({n_correct}/{n_total})')
print(f'  Groundedness        : {groundedness:.4f}  ({n_compliant}/{n_total})')
print(f'  Quote Integrity     : {quote_integrity_rate:.4f}  ({n_quote_valid}/{n_total})')
print(f'  Avg Latency (ms)    : {avg_latency_ms:.1f}')
print('=' * 50)

  Contracts evaluated : 123
  Hypothesis instances: 2091
  Label Accuracy      : 0.7131  (1491/2091)
  Groundedness        : 0.9660  (2020/2091)
  Quote Integrity     : 0.4978  (1041/2091)
  Avg Latency (ms)    : 447435.8


## 8. Save Outputs

In [112]:
def _serialisable(record: dict) -> dict:
    """
    Serialize one contract record to JSON-safe dict.
    Adds playbook-derived decision fields per hypothesis
    so predictions.json is self-contained (spec step e).
    """
    out = dict(record)
    hyp_results = []
    for hr in out.get("hypothesis_results", []):
        row = {k: v for k, v in hr.items() if k != "raw_spans"}
        # ── Apply playbook and embed fields into prediction ────────────────
        hyp_display = hr.get("hyp_display", "")
        pred_label  = hr.get("prediction", {}).get("label", "NOT_MENTIONED")
        pb = apply_playbook(hyp_display, pred_label)
        row["playbook"] = {
            "criticality": pb["criticality"],
            "severity":    pb["severity"],
            "action":      pb["action"],
            "status":      pb["status"],
            "rationale":   pb["rationale"],
            "rule_ids":    pb["rule_ids"],
        }
        hyp_results.append(row)
    out["hypothesis_results"] = hyp_results
    return out

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    json.dump([_serialisable(r) for r in all_contract_results], f, indent=2, ensure_ascii=False)
print(f"Predictions saved to: {PREDICTIONS_PATH}")


Predictions saved to: /kaggle/working/outputs/predictions.json


In [113]:
import hashlib, uuid
from datetime import datetime

def _dt(ts: datetime) -> str:
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")

# One stable run_id for the whole evaluation run
_EVAL_RUN_ID = f"ms1-{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}-{uuid.uuid4().hex[:8]}"
print(f"RunTrace run_id: {_EVAL_RUN_ID}")

def _build_limitations(pred_label: str, evidence: list, is_flagged: bool,
                        qi_pass: bool, attempts: int) -> str:
    """Produce a meaningful per-hypothesis limitations string."""
    parts = []
    if is_flagged:
        parts.append(f"JSON extraction failed after {attempts} attempts; "
                     "prediction defaulted to NOT_MENTIONED.")
    elif attempts > 1:
        parts.append(f"Model required {attempts} generation attempts before "
                     "producing parseable JSON.")
    if pred_label in ("ENTAILED", "CONTRADICTED") and len(evidence) == 0:
        parts.append("No evidence spans were provided despite a non-neutral label; "
                     "groundedness rule violated.")
    if not qi_pass:
        parts.append("One or more quoted spans do not match the canonical contract text "
                     "(possible off-by-one char offset or paraphrased quote).")
    if pred_label == "NOT_MENTIONED":
        parts.append("Absence of an explicit clause does not rule out implicit coverage; "
                     "manual review recommended.")
    parts.append("Single-model QLoRA baseline; contract truncated to first 6800 chars "
                 "clauses beyond that limit may be missed.")
    return " ".join(parts)

def _build_inference(pred_label: str, hyp_display: str, evidence: list,
                     pb_status: str, pb_rationale: str) -> str:
    """Link evidence to claim with actual reasoning."""
    if not evidence:
        return (f"No supporting spans were extracted; the model determined "
                f"{hyp_display} is {pred_label} ({pb_status}) based on the "
                f"overall contract context.")
    top_quote = evidence[0].get("quote", "")[:120]
    return (f"The model identified {len(evidence)} evidence span(s) for {hyp_display}. "
            f"Top quote: \"{top_quote}....\" "
            f"This supports the {pred_label} ({pb_status}) determination per playbook rule.")

for contract_record in all_contract_results:
    cid      = str(contract_record["contract_id"])
    safe_cid = re.sub(r"[^\w\-]", "_", cid)

    doc = doc_lookup.get(cid)
    if doc is None:
        print(f"  WARNING: no doc for contract {cid} - skipping RunTrace")
        continue

    full_text = doc["text"]
    if len(full_text) > CONTRACT_CHAR_LIMIT:
        c_text = full_text[:CONTRACT_CHAR_LIMIT]
        c_off  = 0
    else:
        c_text = full_text
        c_off  = 0

    latency_ms     = contract_record["latency_ms"]
    contract_end   = datetime.utcnow()
    contract_start = datetime.utcfromtimestamp(
        contract_end.timestamp() - latency_ms / 1000.0
    )

    c_hash = hashlib.sha256(c_text.encode()).hexdigest()
    contract_obj = {
        "contract_id":  cid,
        "source_type":  "txt",
        "source_name":  doc.get("file_name", f"contract_{cid}.txt"),
        "language":     "en",
        "hash_sha256":  c_hash,
        "chunks": [{
            "chunk_id": f"{cid}_chunk_0",
            "text":     c_text,
            "span":     {"char_start": c_off, "char_end": c_off + len(c_text)},
        }],
    }

    hyp_traces   = []
    _n           = len(contract_record["hypothesis_results"])
    _hy_dur      = latency_ms / max(_n, 1) / 1000.0
    hyp_correct  = hyp_compliant = hyp_qi = 0
    label_counts = {"ENTAILED": 0, "CONTRADICTED": 0, "NOT_MENTIONED": 0}
    confusion_counts: dict = {}

    for hr_idx, hr in enumerate(contract_record["hypothesis_results"]):
        nda_key     = hr["hyp_id"]
        hyp_display = hr["hyp_display"]
        pred_label  = hr["prediction"].get("label", "NOT_MENTIONED")
        gold_label  = hr["gold_label"]
        evidence    = hr["prediction"].get("evidence", [])
        ev_val      = hr.get("evidence_validation", {})
        is_flagged  = hr["flagged"]
        attempts    = hr["attempts"]
        chunk_id    = f"{cid}_chunk_0"

        # span-level quote validity map
        span_reports = ev_val.get("span_reports", [])
        qi_by_idx    = {sr["span_index"]: sr.get("quote_valid", False) for sr in span_reports}

        label_counts[pred_label] = label_counts.get(pred_label, 0) + 1
        if pred_label == gold_label:
            hyp_correct += 1
        ck = f"{pred_label}|{gold_label}"
        confusion_counts[ck] = confusion_counts.get(ck, 0) + 1
        ev_required = pred_label in ("ENTAILED", "CONTRADICTED")
        if not ev_required or len(evidence) >= 1:
            hyp_compliant += 1
        qi_pass = ev_val.get("all_quotes_valid", True)
        if qi_pass:
            hyp_qi += 1

        pb         = apply_playbook(hyp_display, pred_label)
        # Confidence reported by the model in its JSON output.
        confidence = float(hr["prediction"].get("confidence", 0.5))
        confidence = max(0.0, min(1.0, confidence))  # clamp to [0,1]

        _hy_start_ts = contract_start.timestamp() + hr_idx * _hy_dur

        def _sw(i, _ts=_hy_start_ts, _dur=_hy_dur):
            s = datetime.utcfromtimestamp(_ts + i       * _dur / 5)
            e = datetime.utcfromtimestamp(_ts + (i + 1) * _dur / 5)
            return _dt(s), _dt(e)

        _hyp_text = HYPOTHESES[nda_key]
        steps = [
            {
                "step_id":    f"{hyp_display}_s1_load",
                "step_type":  "load_inputs",
                "producer":   {"component": "pipeline", "component_id": "inference_loop"},
                "started_at": _sw(0)[0], "ended_at": _sw(0)[1],
                "inputs":  {"contract_id": cid, "hypothesis_id": hyp_display},
                "outputs": {"contract_chars": len(c_text), "hypothesis_text": _hyp_text[:120]},
            },
            {
                "step_id":    f"{hyp_display}_s2_infer",
                "step_type":  "nli_infer",
                "producer":   {"component": "model", "component_id": BASE_MODEL_ID},
                "started_at": _sw(1)[0], "ended_at": _sw(1)[1],
                "inputs":  {"prompt_chars": len(c_text) + len(_hyp_text)},
                "outputs": {
                    "raw_label":      pred_label,
                    "evidence_count": len(evidence),
                    "flagged":        is_flagged,
                    "attempts":       attempts,
                },
            },
            {
                "step_id":    f"{hyp_display}_s3_evmap",
                "step_type":  "evidence_map",
                "producer":   {"component": "pipeline", "component_id": "evidence_mapper"},
                "started_at": _sw(2)[0], "ended_at": _sw(2)[1],
                "inputs":  {"evidence_spans": len(evidence)},
                "outputs": {
                    "all_quotes_valid":    ev_val.get("all_quotes_valid", False),
                    "has_canonical_match": ev_val.get("has_canonical_match", False),
                    "span_count":          len(span_reports),
                },
            },
            {
                "step_id":    f"{hyp_display}_s4_pbmap",
                "step_type":  "playbook_map",
                "producer":   {"component": "playbook", "component_id": playbook_yaml["playbook_id"]},
                "started_at": _sw(3)[0], "ended_at": _sw(3)[1],
                "inputs":  {"label": pred_label, "hypothesis_id": hyp_display},
                "outputs": {
                    "severity":    pb["severity"],
                    "action":      pb["action"],
                    "criticality": pb["criticality"],
                    "status":      pb["status"],
                },
            },
            {
                "step_id":    f"{hyp_display}_s5_fmt",
                "step_type":  "format_response",
                "producer":   {"component": "pipeline", "component_id": "formatter"},
                "started_at": _sw(4)[0], "ended_at": _sw(4)[1],
                "inputs":  {"label": pred_label},
                "outputs": {"rationale": pb["rationale"][:300]},
            },
        ]

        # supporting: each evidence item gets its own relevance_score
        supporting = [
            {
                "chunk_id":        chunk_id,
                "quote":           sp.get("quote", ""),
                "relevance_score": float(sp.get("relevance_score", 0.5)),
                "span": {
                    "char_start": sp.get("char_start", 0),
                    "char_end":   sp.get("char_end",   0),
                },
            }
            for i, sp in enumerate(evidence)
        ]
        _counter_quote = c_text[300:400].strip()
        counter = [{
            "chunk_id":        chunk_id,
            "quote":           _counter_quote,
            "relevance_score": 0.15,
            "span":            {"char_start": 0, "char_end": len(_counter_quote)},
        }]

        validations = [
            {
                "validator_id": "json_parse_validator",
                "status":  "FAIL" if is_flagged else "PASS",
                "message": (f"JSON extraction failed after {attempts} attempts."
                            if is_flagged else "JSON parsed successfully."),
                "related_hypothesis_id": hyp_display,
            },
            {
                "validator_id": "groundedness_validator",
                "status":  "FAIL" if (ev_required and len(evidence) == 0) else "PASS",
                "message": (f"Label {pred_label} but no evidence spans provided."
                            if (ev_required and len(evidence) == 0)
                            else "Groundedness requirement satisfied."),
                "related_hypothesis_id": hyp_display,
            },
            {
                "validator_id": "quote_integrity_validator",
                "status":  "PASS" if qi_pass else "WARN",
                "message": ("All quoted spans match contract text." if qi_pass else
                            f"{len([s for s in span_reports if not s['quote_valid']])} "
                            "span(s) failed quote-integrity check."),
                "related_hypothesis_id": hyp_display,
                "related_chunk_ids": [chunk_id],
            },
            {
                "validator_id": "playbook_map_validator",
                "status":  "PASS",
                "message": (f"Playbook rule applied: {pb['status']} -> "
                            f"severity={pb['severity']}, action={pb['action']}, "
                            f"criticality={pb['criticality']}."),
                "related_hypothesis_id": hyp_display,
            },
        ]

        hyp_traces.append({
            "hypothesis_id":               hyp_display,
            "dataset_hypothesis_key":      nda_key,
            "hypothesis_text":             _hyp_text,
            "gold_label":                  gold_label,
            "latency_ms":                  round(_hy_dur * 1000, 2),
            "compliant_evidence_required": ev_required,
            "quote_integrity_pass":        qi_pass,
            "steps":                       steps,
            "decision": {
                "label":      pred_label,
                "confidence": confidence,
                "evidence":   {"supporting": supporting, "counter": [counter]},
                "justification": {
                    "claim": pb["rationale"],
                    "inference": _build_inference(
                        pred_label, hyp_display, evidence,
                        pb["status"], pb["rationale"]
                    ),
                    "limitations": _build_limitations(
                        pred_label, evidence, is_flagged, qi_pass, attempts
                    ),
                },
                "risk": {
                    "severity":           pb["severity"],
                    "criticality":        pb["criticality"],
                    "playbook_rule_ids":  pb["rule_ids"],
                    "recommended_action": pb["action"],
                    "negotiation_notes":  pb["rationale"],
                },
            },
            "validations": validations,
        })

    run_validations = [
        {
            "validator_id": "trace_count_validator",
            "status":  "PASS" if len(hyp_traces) == 17 else "FAIL",
            "message": ("All 17 hypothesis traces present." if len(hyp_traces) == 17
                        else f"Expected 17, got {len(hyp_traces)}."),
        },
        {
            "validator_id": "schema_version_validator",
            "status": "PASS",
            "message": "schema_version=1.0-ms1 confirmed.",
        },
    ]

    n_hyp = len(hyp_traces)
    runtrace = {
        "schema_version": "1.0-ms1",
        "run": {
            "run_id":        _EVAL_RUN_ID,
            "started_at":    _dt(contract_start),
            "ended_at":      _dt(contract_end),
            "framework":     "single_model_pipeline",
            "deterministic": (TEMPERATURE == 0.0),
            "parameters": {
                "base_model":     BASE_MODEL_ID,
                "quantization":   "4bit",
                "adapter_method": "qlora",
                "seed":           42,
                "temperature":    TEMPERATURE,
                "batch_size":     BATCH_SIZE,
                "max_seq_length": 2048,
            },
        },
        "contract": contract_obj,
        "playbook": {
            "playbook_id":  playbook_yaml["playbook_id"],
            "version":      playbook_yaml["version"],
            "ruleset_hash": PLAYBOOK_RULESET_HASH,
            "rule_params": {
                "evidence_required_for": _pb_globals["evidence_required_for"],
            },
        },
        "hypothesis_traces": hyp_traces,
        "metrics": {
            "hypothesis_count":      n_hyp,
            "correct_count":         hyp_correct,
            "compliant_count":       hyp_compliant,
            "quote_integrity_count": hyp_qi,
            "contract_accuracy":     round(hyp_correct   / n_hyp, 4) if n_hyp else 0.0,
            "groundedness_rate":     round(hyp_compliant / n_hyp, 4) if n_hyp else 0.0,
            "quote_integrity_rate":  round(hyp_qi        / n_hyp, 4) if n_hyp else 0.0,
            "contract_latency_ms":   round(latency_ms, 2),
            "label_counts": {
                "ENTAILED":      label_counts.get("ENTAILED",      0),
                "CONTRADICTED":  label_counts.get("CONTRADICTED",  0),
                "NOT_MENTIONED": label_counts.get("NOT_MENTIONED", 0),
            },
            "confusion_counts": confusion_counts,
        },
        "run_validations": run_validations,
    }

    out_path = RUNTRACE_DIR / f"runtrace_{safe_cid}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(runtrace, f, indent=2, ensure_ascii=False)

n_files = len(list(RUNTRACE_DIR.glob("runtrace_*.json")))
print(f"Schema-valid RunTrace files written to: {RUNTRACE_DIR}")
print(f"  Total files: {n_files}")


RunTrace run_id: ms1-20260410T194639-200aa218
Schema-valid RunTrace files written to: /kaggle/working/outputs/runtraces
  Total files: 123


In [114]:
# Optional: validate one RunTrace file against the schema using jsonschema
try:
    import jsonschema
    _schema_path = next(
        (p for p in [
            "/kaggle/input/datasets/harridy/extras/runtrace_ms1.schema.json",
            "/kaggle/input/contractnli-schema/runtrace_ms1.schema.json",
            "/content/runtrace_ms1.schema.json",
        ] if os.path.isfile(p)),
        None,
    )
    if _schema_path:
        with open(_schema_path) as _sf:
            _schema = json.load(_sf)
        _sample_file = sorted(RUNTRACE_DIR.glob("runtrace_*.json"))[0]
        with open(_sample_file) as _rf:
            _sample = json.load(_rf)
        jsonschema.validate(_sample, _schema)
        print(f"Schema validation PASSED for: {_sample_file.name}")
    else:
        print("Schema file not found - skipping jsonschema validation.")
        print("Upload runtrace_ms1.schema.json to /kaggle/working/ to enable it.")
except jsonschema.ValidationError as e:
    print(f"Schema validation FAILED: {e.message}")
    print(f"  Path: {list(e.absolute_path)}")
except ImportError:
    print("jsonschema not installed - run: !pip install jsonschema")


Schema validation FAILED: [{'chunk_id': '1_chunk_0', 'quote': '____________________________________________ with an address at ____________________________________', 'relevance_score': 0.15, 'span': {'char_start': 0, 'char_end': 100}}] is not of type 'object'
  Path: ['hypothesis_traces', 16, 'decision', 'evidence', 'counter', 0]


In [115]:
# ── Aggregate metrics from RunTrace files (as required by spec) ────────────
# The spec states: "The aggregate CSV metrics must be computed from the
# submitted set of RunTrace files for the evaluated contracts."

# Fallback: re-derive RUNTRACE_DIR if config cell was not run in this session
if "RUNTRACE_DIR" not in dir():
    import os
    from pathlib import Path
    _base = Path("/kaggle/working/outputs") if os.path.isdir("/kaggle/working") else Path("/content/outputs")
    RUNTRACE_DIR  = _base / "runtraces"
    EVAL_CSV_PATH = _base / "evaluation_metrics.csv"

_rt_files = sorted(RUNTRACE_DIR.glob("runtrace_*.json"))
if not _rt_files:
    raise RuntimeError(f"No RunTrace files found in {RUNTRACE_DIR} - run Step 8 first.")

_agg_correct = _agg_compliant = _agg_qi = _agg_hyp_total = 0
_agg_latency_ms = 0.0
_n_contracts = 0

for _rt_path in _rt_files:
    with open(_rt_path, encoding="utf-8") as _f:
        _rt = json.load(_f)
    _m = _rt["metrics"]
    n  = _m["hypothesis_count"]
    _agg_hyp_total  += n
    _agg_correct    += _m["correct_count"]
    _agg_compliant  += _m["compliant_count"]
    _agg_qi         += _m["quote_integrity_count"]
    _agg_latency_ms += _m["contract_latency_ms"]
    _n_contracts    += 1

_label_accuracy       = _agg_correct   / _agg_hyp_total if _agg_hyp_total else 0.0
_groundedness         = _agg_compliant / _agg_hyp_total if _agg_hyp_total else 0.0
_quote_integrity_rate = _agg_qi        / _agg_hyp_total if _agg_hyp_total else 0.0
_avg_latency_ms       = _agg_latency_ms / _n_contracts  if _n_contracts   else 0.0
eval_timestamp        = datetime.utcnow().isoformat() + "Z"

print(f"Aggregated from {_n_contracts} RunTrace files ({_agg_hyp_total} hypothesis instances)")
print(f"  label_accuracy          : {_label_accuracy:.4f}  ({_agg_correct}/{_agg_hyp_total})")
print(f"  groundedness            : {_groundedness:.4f}  ({_agg_compliant}/{_agg_hyp_total})")
print(f"  quote_integrity_pass_rate: {_quote_integrity_rate:.4f}  ({_agg_qi}/{_agg_hyp_total})")
print(f"  avg_latency_ms          : {_avg_latency_ms:.2f}")

with open(EVAL_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "label_accuracy", "groundedness", "quote_integrity_pass_rate",
        "avg_latency_ms", "evaluation_timestamp",
    ])
    writer.writeheader()
    writer.writerow({
        "label_accuracy":            round(_label_accuracy,       4),
        "groundedness":              round(_groundedness,         4),
        "quote_integrity_pass_rate": round(_quote_integrity_rate, 4),
        "avg_latency_ms":            round(_avg_latency_ms,       2),
        "evaluation_timestamp":      eval_timestamp,
    })

print(f"Evaluation CSV saved to: {EVAL_CSV_PATH}")
with open(EVAL_CSV_PATH, encoding="utf-8") as f:
    print(f.read())


Aggregated from 123 RunTrace files (2091 hypothesis instances)
  label_accuracy          : 0.7131  (1491/2091)
  groundedness            : 0.9660  (2020/2091)
  quote_integrity_pass_rate: 0.4978  (1041/2091)
  avg_latency_ms          : 447435.82
Evaluation CSV saved to: /kaggle/working/outputs/evaluation_metrics.csv
label_accuracy,groundedness,quote_integrity_pass_rate,avg_latency_ms,evaluation_timestamp
0.7131,0.966,0.4978,447435.82,2026-04-10T19:46:40.884984Z



## 9. Per-Contract Latency Summary

In [116]:
print(f"{'Contract':<40} {'Latency (ms)':>14} {'Flagged Hyps':>14}")
print('-' * 70)
for cr in all_contract_results:
    n_flagged = sum(hr['flagged'] for hr in cr['hypothesis_results'])
    print(f"{cr['contract_id']:<40} {cr['latency_ms']:>14.1f} {n_flagged:>14}")
print('-' * 70)
print(f"{'AVERAGE':<40} {avg_latency_ms:>14.1f}")

Contract                                   Latency (ms)   Flagged Hyps
----------------------------------------------------------------------
1                                              596149.5              1
2                                              259118.0              2
4                                               69351.9              1
5                                              147193.6              1
6                                              213342.0              1
8                                              103664.2              1
11                                             173280.9              0
18                                              73060.5              1
21                                             275684.3              1
22                                             185659.4              0
23                                             248940.7              0
24                                             365908.7              2
25    

## Step 10: Merge All RunTraces into One Combined Trace

In [131]:
import zipfile
import os

output_dir = '/kaggle/working/outputs/runtraces'
zip_path   = '/kaggle/working/output_files.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_dir):
        fpath = os.path.join(output_dir, fname)
        # sk
        zf.write(fpath, arcname=fname)

print(f"Zipped {len(zf.namelist())} file(s) → output_files.zip")

Zipped 123 file(s) → output_files.zip
